In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [12]:
## load the dataset

data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [13]:
## Preprocess the data
## Drop the unnecessary columns
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [21]:
##  encode the categorical variables

le_gender=LabelEncoder()
data['Gender']=le_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [8]:
## One hot encoding for the 'Geography' column
## using get_dummies to create dummy variables for the 'Geography' column and dropping the first category to avoid multicollinearity
## pandas implementation of one hot encoding
data=pd.get_dummies(data,columns=['Geography'],drop_first=True)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,False,False
1,608,0,41,1,83807.86,1,0,1,112542.58,0,False,True
2,502,0,42,8,159660.80,3,1,0,113931.57,1,False,False
3,699,0,39,1,0.00,2,0,0,93826.63,0,False,False
4,850,0,43,2,125510.82,1,1,1,79084.10,0,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,False,False
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,False,False
9997,709,0,36,7,0.00,1,0,1,42085.58,1,False,False
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,True,False


In [14]:
## One hot encoding for the 'Geography' column
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
geo_encoder = encoder.fit_transform(data[['Geography']])
feature_names = encoder.get_feature_names_out(['Geography'])

print(feature_names)
print(geo_encoder)

['Geography_France' 'Geography_Germany' 'Geography_Spain']
[[1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 ...
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]]


In [16]:
geo_encoded_df = pd.DataFrame(geo_encoder, columns=feature_names)
geo_encoded_df


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [ ]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [17]:
## combine the encoded columns with the original dataframe

data=pd.concat([data.drop(columns=['Geography']), geo_encoded_df], axis=1)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,Female,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,Female,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,Female,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,Female,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,Female,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,Male,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,Male,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,Female,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,Male,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [87]:
## save the encoder and scalar
with open('onehot_encoder_geo.pkl', 'wb') as f:
    pickle.dump(encoder, f)
with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(le_gender, f)

In [18]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,Female,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,Female,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,Female,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,Female,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,Female,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [79]:
## divide the data into features (independent variables) and target variable(dependent variable)
X=data.drop('Exited',axis=1)
y=data['Exited']

## split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)



In [80]:
X_train.shape

(8000, 12)

                  Original data    
                       │    
                 ┌─────┴─────┐    
                 ↓           ↓    
              80%           20%  
             training       test  
                 │  
          split it again  
                 │  
            ┌────┴────┐  
            ↓         ↓  
          80%         20%  
        training    validation  

In [81]:
X_train, x_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=0
)

In [82]:
X_train.shape,x_val.shape,y_train.shape,y_val.shape

((6400, 12), (1600, 12), (6400,), (1600,))

In [83]:
## Scale the features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
x_val = scaler.transform(x_val)
X_test = scaler.transform(X_test)


In [84]:
X_train.shape,x_val.shape,y_train.shape,y_val.shape

((6400, 12), (1600, 12), (6400,), (1600,))

In [85]:
X_train

array([[-1.1302282 ,  0.9148601 ,  0.00212983, ...,  0.97530483,
        -0.5658021 , -0.56965192],
       [ 1.3106727 ,  0.9148601 , -1.42768856, ...,  0.97530483,
        -0.5658021 , -0.56965192],
       [-1.71728032, -1.0930633 , -0.37915507, ..., -1.02532046,
        -0.5658021 ,  1.75545796],
       ...,
       [ 0.00267939,  0.9148601 , -1.52300979, ...,  0.97530483,
        -0.5658021 , -0.56965192],
       [-0.04881641,  0.9148601 ,  1.05066333, ...,  0.97530483,
        -0.5658021 , -0.56965192],
       [-1.439203  ,  0.9148601 ,  0.57405719, ...,  0.97530483,
        -0.5658021 , -0.56965192]])

In [86]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [24]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


## ANN Architecture

1. BUILD   
   ↓  
   model = Sequential(...)  

2. COMPILE  
   ↓  
   model.compile(...)  
   
3. TRAIN   
   ↓  
   model.fit(...)  
  
4. PREDICT  
   ↓  
   model.predict(...)  

## ANN Implementation

In [31]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [26]:
X_train.shape

(8000, 12)

In [27]:
X_train.shape[0], X_train.shape[1]

(8000, 12)

In [28]:
X_train.shape[1]

12

Why can't the input just be 1D?

Because the model needs to know where one sample ends and the next sample begins.

reshape(1, -1) is mainly useful when you have a single sample stored as a 1D array and need to add the sample/batch dimension.

In [29]:
X_train.shape[1],

(12,)

## 1. BUILD

input_shape expects a tuple describing the shape of ONE sample.

In [62]:
model_1=Sequential([
    
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)), ##HL1 Connected to input layer
    Dense(16, activation='relu'), ## HL2 Connected to HL1
    Dense(units=1, activation='sigmoid') ## Output layer connected to HL2
]

)

In [66]:
## clear Keras' naming state without restarting the whole kernel:
tf.keras.backend.clear_session()

In [63]:
model_1.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                416       
                                                                 
 dense_1 (Dense)             (None, 16)                528       
                                                                 
 dense_2 (Dense)             (None, 1)                 17        
                                                                 
Total params: 961 (3.75 KB)
Trainable params: 961 (3.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [67]:
## Build the ANN model
model_2= Sequential()
model_2.add(Dense(units=64, activation='relu', input_dim=12))
model_2.add(Dense(units=32, activation='relu'))
model_2.add(Dense(units=1, activation='sigmoid'))

In [69]:
## clear Keras' naming state without restarting the whole kernel:
tf.keras.backend.clear_session()

In [68]:
model_2.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## 2. COMPILE

In [95]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()

In [96]:
## compile the model
model_2.compile(optimizer=opt, loss="binary_crossentropy", metrics=['accuracy'])

Compile model  
      ↓  
Create log directory  
      ↓  
Create TensorBoard callback  
      ↓  
model.fit()  
      ↓  
Training information gets saved  
      ↓  
TensorBoard reads those logs  
      ↓  
You see graphs/dashboard   

### 3. TRAIN

In [115]:
## set up TensorBoard callback
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [116]:
## set up TensorBoard callback
tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

In [117]:
## set up EarlyStopping callback to prevent overfitting and stop training when the validation loss stops improving
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

## Why have patience at all?

Because validation loss can temporarily get worse and then improve again.

For example:

Epoch 3 → 0.40  ← best  
Epoch 4 → 0.42 
Epoch 5 → 0.41    
Epoch 6 → 0.38  ← NEW BEST!  

If we stopped immediately at epoch 4, we'd have stopped too early.

patience=5 gives the model some breathing room:

"Don't stop just because things got slightly worse. Give it 5 more epochs to see if it recovers."

TensorBoard callback = record something when Keras calls it

EarlyStopping callback = check something when Keras calls it

In [118]:
model_history = model_2.fit(
    X_train,
    y_train,
    epochs=100,
    validation_data=(x_val, y_val),
    callbacks=[tensorboard_callback, early_stopping]
)

Epoch 1/100
200/200 [==============================] - 1s 3ms/step - loss: 0.2912 - accuracy: 0.8783 - val_loss: 9836.1621 - val_accuracy: 0.6294
Epoch 2/100
200/200 [==============================] - 1s 3ms/step - loss: 0.2863 - accuracy: 0.8791 - val_loss: 3532.7256 - val_accuracy: 0.7219
Epoch 3/100
200/200 [==============================] - 1s 4ms/step - loss: 0.2885 - accuracy: 0.8777 - val_loss: 960.9777 - val_accuracy: 0.7431
Epoch 4/100
200/200 [==============================] - 1s 4ms/step - loss: 0.2873 - accuracy: 0.8788 - val_loss: 6784.8125 - val_accuracy: 0.5394
Epoch 5/100
200/200 [==============================] - 1s 6ms/step - loss: 0.2858 - accuracy: 0.8772 - val_loss: 5710.0142 - val_accuracy: 0.5406
Epoch 6/100
200/200 [==============================] - 1s 4ms/step - loss: 0.2810 - accuracy: 0.8800 - val_loss: 14848.0771 - val_accuracy: 0.4550
Epoch 7/100
200/200 [==============================] - 1s 4ms/step - loss: 0.2778 - accuracy: 0.8814 - val_loss: 15470.1533 

In [119]:
model_2.save('model.h5')

e:\ANN classification\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


means:

Save the model in an HDF5 (Hierarchical Data Format version 5) file.

model_history = model.fit(...)

→ saves what happened during training

model.save('model.h5')

→ saves the actual trained model.

                     model.fit()
                            │
              ┌─────────────┴─────────────┐
              ↓                           ↓
       TensorBoard callback          EarlyStopping callback
              ↓                           ↓
       Record training              Watch val_loss
       information                      ↓
              │                  Is it improving?
              │                    ↙          ↘
              │                  YES          NO
              │                   ↓            ↓
              │                continue    patience...
              │                                ↓
              │                         still not improving?
              │                                ↓
              │                              STOP
              ↓
       View training in
       TensorBoard

Callback → a helper that watches/acts during training.  
Frequency → how often a callback records/checks something.  
Early stopping → automatically stop training when the model stops improving according to the metric you're monitoring.  

## Better pipeline when using validation

                    DATA
                      ↓
              X and y separated
                      ↓
              ┌───────┴───────┐
              ↓               ↓
          80% train         20% test
              ↓               ↓
       split again             │
              ↓               │
       ┌──────┴──────┐         │
       ↓             ↓         │
     64% train      16% val    20% test
       ↓             ↓         ↓
     scaler          scaler    scaler
     fit + transform transform transform
       ↓             ↓         ↓
       └─────────────┴─────────┘
                      ↓
                  model.fit()
                      ↓
              train → learn weights
              val   → monitor
                      ↓
                  final model
                      ↓
                  test once

## Training data

Used to learn weights.

## Validation data

Used during development to make decisions about the model:

How many epochs?  
Which architecture?  
Which hyperparameters?  
Is it overfitting?  

## Test data  

Saved until the end to answer: 
"Now that we've finished making all our decisions, how well does the final model actually perform on completely held-out data?"

## Validation data is basically your development-time examiner.

                  TRAINING
                        ↓
                  Update weights
                        ↓
                  Model
                        ↓
                  VALIDATION
                        ↓
            "How well are you doing?"
                        ↓
            ┌────────────┴────────────┐
            ↓                         ↓
      Good result              Bad result
            ↓                         ↓
      Continue / choose        Change architecture /
      this model               hyperparameters / epochs

In [120]:

## Load TensorBoard Extension in Jupyter Notebook
%load_ext tensorboard   

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [121]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 21624), started 0:07:34 ago. (Use '!kill 21624' to kill it.)

### 4. PREDICT